# 멘디 파손 검출 — RF-DETR 학습과 YOLO26 비교 (Colab GPU)

RF-DETR(Roboflow, **Apache-2.0** — 상용 앱에 라이선스 비용 없이 넣을 수 있음)로 파손 박스(**outsole** = 밑창 벌어짐,
**heel lining** = 뒤꿈치 안감 손상)를 학습하고, YOLO26 노트북과 **같은 사진 · 같은 정답 · 같은 train/val 분리 · 같은 채점 코드**로 비교합니다.

**실행 순서**: 런타임 → 런타임 유형 변경 → **T4 GPU** → 런타임 → **모두 실행**. 3번 셀에서 파일을 올리라고 하면 한꺼번에 골라 올립니다.

**올릴 파일 3개** — YOLO 노트북과 같습니다 (이미 `MyDrive/mendii/yolo26_damage/`에 있으면 올리지 않아도 됩니다)
- CVAT 프로젝트 최종 export — `mendii_shoes_20261014_0900.zip`
- 밑창 재라벨 — `outsole_relabel.json`
- S3 읽기 전용 키 — `mendii-reader_accessKeys.csv`

**YOLO 노트북을 먼저 돌렸다면** 같은 Drive 폴더의 사진 캐시(`img1280.zip`)와 분리 목록(`runs/yolo26s_1024_final_e60/split.json`)을 그대로 씁니다.
그래서 학습·검증 사진이 YOLO와 한 장도 다르지 않고, 9번 셀이 YOLO 모델(`last.pt`)도 같은 코드로 다시 채점해 나란히 보여 줍니다.

**공정한 비교를 위해**: YOLO처럼 학습 중에는 검증하지 않고(마지막 epoch에 한 번만) 마지막 EMA 가중치(`last_ema.pth`)로 채점합니다.
검증 사진으로 가장 좋은 epoch를 고르지 않습니다. 결과는 Drive `MyDrive/mendii/yolo26_damage/runs_rfdetr/`에 저장됩니다.
사진에 작업지시서(개인정보)가 찍혀 있으니 결과 그림이 남은 노트북은 남에게 공유하지 마세요.

In [ ]:
# 1. 런타임 확인. CUDA가 False면 런타임 > 런타임 유형 변경에서 GPU(T4)를 고르고 다시 실행하세요.
import torch

print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU로는 학습이 사실상 불가능합니다. GPU 런타임으로 바꾸세요.")

In [ ]:
# 2. 설치. 버전을 고정해 두어야 나중에 다시 돌려도 같은 결과가 나옵니다.
# ultralytics는 YOLO 모델을 같은 코드로 다시 채점할 때만 씁니다(앱에 넣는 것은 RF-DETR).
!pip install -q "rfdetr[train]==1.11.2" "ultralytics==8.4.169" boto3

import importlib.metadata as md

import pytorch_lightning  # noqa: F401  학습용 추가 패키지가 깔렸는지 여기서 바로 확인합니다.
print("rfdetr", md.version("rfdetr"), "| torch", md.version("torch"))
if md.version("torch").split("+")[0] != torch.__version__.split("+")[0]:
    print("⚠️ 설치하면서 torch 버전이 바뀌었습니다. 런타임 > 세션 다시 시작을 누른 뒤 다시 모두 실행하세요.")

In [ ]:
# 3. 설정과 입력 파일. 바꿀 것은 이 셀 위쪽의 대문자 값뿐입니다.
DRIVE_DIR   = "MyDrive/mendii/yolo26_damage"   # 결과·사진 캐시를 저장할 Drive 폴더
BUCKET      = "mendii-sneakers"
REGION      = "ap-northeast-2"
STORE_SIZE  = 1280      # S3 원본(3456px)을 이 크기로 줄여 저장합니다.
REVIEW_GAP  = 100       # 박스 달린 사진 사이가 이 장수보다 멀면 그 사이는 아직 검수 안 한 것으로 봅니다.
SESSION_GAP = 300       # 촬영 간격이 이 초 이하면 같은 신발(같은 세션)로 봅니다.
VAL_RATIO   = 0.2       # 검증으로 뺄 세션 비율
NEG_PER_POS = 1.0       # 학습에 넣을 '파손 없음' 사진 수 = 파손 사진 수 × 이 값 (검증에는 전부 넣음)
REPEAT_OUTSOLE = 2      # outsole(밑창 벌어짐) 사진을 학습에 몇 번 넣을지. 드문 클래스라 2번 넣습니다(LVIS식 반복 샘플링).
SEED        = 0

# 학습. 안감 파손 박스가 작아서(중앙값이 사진 한 변의 7%) 기본 해상도(512)보다 키웁니다. 32의 배수여야 합니다.
RF_MODEL   = "small"       # small(기본, 학습 빠름) / medium(조금 더 정확, 약 1.4배 느림)
RESOLUTION = 768           # 896·1024도 되지만 T4에서 느려집니다(해상도 제곱에 비례).
EPOCHS     = 20            # 8번 셀이 첫 epoch 시간을 보여 줍니다. 너무 길면 멈추고 줄이세요.
NAME       = "rfdetr_s_768_e20"   # 결과 폴더 이름. 설정을 바꿔 다시 학습하면 이름도 바꾸세요.
RESUME     = False         # 끊긴 학습을 이어 갈 때 True로 바꾸고 모두 실행
YOLO_RUN   = "yolo26s_1024_final_e60"   # 비교할 YOLO 학습(같은 Drive 폴더의 runs/). 그 분리 목록과 last.pt를 씁니다.
YOLO_IMGSZ = 1024
SPLIT_FROM = "yolo26s_1024"   # YOLO_RUN의 split.json이 없을 때만: YOLO 노트북과 같은 규칙으로 새로 나눔
QUICK_TEST = False         # True면 사진 180장·1 epoch로 노트북 전체가 도는지만 확인합니다.

# 다 검수했다고 확인된 CVAT job (첫 사진, 마지막 사진). 박스가 적어도 이 범위는 검수된 사진입니다.
KNOWN_REVIEWED = {
    "job 24": ("20240819_093048.jpg", "20240902_133744.jpg"),
    "job 25": ("20240903_092559.jpg", "20240913_154100.jpg"),
}

import csv
import io
import json
import os
import sys
import zipfile
from pathlib import Path

if QUICK_TEST:
    RESOLUTION, EPOCHS, NAME = 320, 1, "quick_test"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content")
    OUT_DIR = Path("/content/drive") / DRIVE_DIR
else:                                   # Colab 밖(로컬 GPU 등)에서 돌릴 때
    ROOT = Path(os.environ.get("MENDII_ROOT", "mendii_yolo26")).resolve()
    OUT_DIR = ROOT / "out"
ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
ANN = "annotations/instances_default.json"


def read_cvat(p):
    """CVAT COCO export면 (export 시각, coco) 아니면 None."""
    try:
        with zipfile.ZipFile(p) as z:
            return z.getinfo(ANN).date_time, json.loads(z.read(ANN))
    except (zipfile.BadZipFile, KeyError, OSError):
        return None


def find_inputs():
    places = [ROOT, OUT_DIR]
    found = {}
    for p in sorted(q for d in places for q in d.glob("*.zip")):
        r = read_cvat(p)
        if r and p.name not in found:
            found[p.name] = (p, *r)
    keys = [p for d in places for p in d.glob("*.csv") if "accesskey" in p.name.lower()]
    return list(found.values()), (max(keys, key=lambda p: p.stat().st_mtime) if keys else None)


def find_relabel():
    ps = [p for d in (ROOT, OUT_DIR) for p in d.glob("outsole_relabel*.json")]
    return max(ps, key=lambda p: p.stat().st_mtime) if ps else None


def secret(name):
    if not IN_COLAB:
        return os.environ.get(name)
    from google.colab import userdata
    try:
        return userdata.get(name)
    except Exception:                    # 비밀이 없거나 이 노트북에 접근 권한을 안 준 경우
        return None


PROJECT_MIN_IMAGES = 5000   # 프로젝트 전체 export는 사진 18,168장, job export는 400여 장입니다.


def missing_inputs(exports, key_csv):
    miss = []
    if not any(len(e[2]["images"]) >= PROJECT_MIN_IMAGES for e in exports):
        miss.append("CVAT 프로젝트 최종 export (mendii_shoes_20261014_0900.zip, 사진 12,599장)")
    if not all(AWS_KEY) and key_csv is None:
        miss.append("S3 키 (mendii-reader_accessKeys.csv)")
    if find_relabel() is None:
        miss.append("밑창 재라벨 (outsole_relabel.json)")
    return miss


AWS_KEY = (secret("AWS_ACCESS_KEY_ID"), secret("AWS_SECRET_ACCESS_KEY"))
exports, key_csv = find_inputs()
for k in range(3):                       # 꼭 필요한 파일(프로젝트 export·키)이 올라올 때까지 다시 묻습니다.
    miss = missing_inputs(exports, key_csv)
    blocking = [m for m in miss if not m.startswith("밑창")]
    if not miss or not IN_COLAB or (k > 0 and not blocking):
        break
    from google.colab import files
    print("올릴 파일 (여러 개를 한꺼번에 골라도 됩니다):\n  - " + "\n  - ".join(miss))
    print("이미 있는 것:", [e[0].name for e in exports] or "없음")
    for name, data in files.upload().items():
        (ROOT / name).write_bytes(data)
    exports, key_csv = find_inputs()
miss = missing_inputs(exports, key_csv)
assert not [m for m in miss if not m.startswith("밑창")], "다음 파일이 없습니다: " + ", ".join(miss)
RELABEL = find_relabel()
print("밑창 재라벨:", RELABEL.name if RELABEL else "⚠️ 없음 — outsole은 CVAT 박스 그대로(기준이 섞여 있음)")

# 사진이 가장 많은 zip = 프로젝트 export(바탕). 그보다 나중에 받은 job export는 그 사진들의 박스를 바꿉니다.
main_path, main_time, coco = max(exports, key=lambda e: (len(e[2]["images"]), e[1]))
overrides = sorted((e for e in exports if e[0] != main_path and e[1] > main_time), key=lambda e: e[1])
ignored = [e[0].name for e in exports if e[0] != main_path and e[1] <= main_time]
fmt = lambda t: "%04d-%02d-%02d %02d:%02d" % t[:5]
print(f"바탕: {main_path.name} (export {fmt(main_time)} UTC) | 사진 {len(coco['images'])} | 박스 {len(coco['annotations'])}"
      f" | 라벨 {[c['name'] for c in coco['categories']]}")
for p, t, c in overrides:
    print(f"덮어쓰기: {p.name} (export {fmt(t)} UTC) | 사진 {len(c['images'])} | 박스 {len(c['annotations'])}")
if ignored:
    print("바탕보다 먼저 받은 export라 쓰지 않음:", ignored)
if not overrides and main_time < (2026, 10, 12, 17, 43, 0):
    print("⚠️ 10/12 이전 export입니다. job 24는 복원 전 상태일 수 있으니 job 24·25 수정본(10/12 export)을 같이 올리세요.")

if not all(AWS_KEY):
    assert key_csv is not None, "S3 키(mendii-reader_accessKeys.csv)를 찾지 못했습니다."
    row = next(csv.DictReader(io.StringIO(key_csv.read_text(encoding="utf-8-sig"))))
    AWS_KEY = (row["Access key ID"].strip(), row["Secret access key"].strip())
    if key_csv.parent == ROOT and IN_COLAB:
        key_csv.unlink()                 # 런타임 디스크에 키를 남기지 않습니다.

In [ ]:
# 4. 검수된 사진 고르기, job export로 덮어쓰기, COCO → YOLO 박스 변환.
from collections import Counter, defaultdict

CLASSES = ["outsole", "heel lining"]                      # YOLO 클래스 0, 1


def labels_by_name(c):
    """file_name -> [(라벨 이름, [x, y, w, h])]. 박스 없는 사진도 빈 목록으로 들어갑니다."""
    cat = {x["id"]: x["name"] for x in c["categories"]}
    ims = {im["id"]: im for im in c["images"]}
    out = {im["file_name"]: [] for im in c["images"]}
    for a in c["annotations"]:
        out[ims[a["image_id"]]["file_name"]].append((cat[a["category_id"]], a["bbox"]))
    return out


size = {im["file_name"]: (im["width"], im["height"]) for im in coco["images"]}
labels = labels_by_name(coco)

# 프로젝트 export의 사진 id 순서 = task·프레임 순서(대체로 촬영 순서). 박스가 달린 사진의 위치로 검수 구간을 추정합니다.
order = [im["file_name"] for im in sorted(coco["images"], key=lambda im: im["id"])]
hit = [k for k, n in enumerate(order) if labels[n]]
runs, start = [], hit[0]
for prev, k in zip(hit, hit[1:]):
    if k - prev > REVIEW_GAP:
        runs.append((start, prev))
        start = k
runs.append((start, hit[-1]))
reviewed = {n for s, e in runs for n in order[s:e + 1]}
for lo, hi in KNOWN_REVIEWED.values():
    reviewed |= {n for n in order if lo <= n <= hi}

# 나중에 받은 job export가 있으면 그 사진들의 박스를 통째로 바꾸고, 검수된 사진으로 봅니다.
for p, t, c in overrides:
    new = labels_by_name(c)
    changed = sum(sorted(map(str, labels.get(n, []))) != sorted(map(str, b)) for n, b in new.items())
    labels.update(new)
    size.update({im["file_name"]: (im["width"], im["height"]) for im in c["images"]})
    reviewed |= set(new)
    print(f"{p.name}: 사진 {len(new)}장 중 {changed}장의 박스가 바뀜")

# 밑창 재라벨: 그 사진들의 outsole 박스를 통째로 바꿉니다(빈 목록 = outsole 없음). heel lining 박스는 그대로 둡니다.
# 재라벨에서 '판단 불가'로 남긴 사진은 학습·채점에서 빼고, '파손 아님'으로 확인한 사진은 5번 셀에서 음성으로 먼저 씁니다.
unsure, hard_neg = set(), {}
if RELABEL:
    rl = json.loads(RELABEL.read_text())
    unsure = set(rl.get("uncertain", []))
    hard_neg = {f: c.split(":")[-1] for f, c in rl.get("category", {}).items() if c.split(":")[-1] in ("deleted", "none")}
    n_box = n_changed = 0
    for n, bs in rl["images"].items():
        if n not in size:
            continue
        W, H = size[n]
        old = sorted(round(v) for name, b in labels.get(n, []) if name == "outsole" for v in b)
        new = [("outsole", [x0 * W, y0 * H, (x1 - x0) * W, (y1 - y0) * H]) for x0, y0, x1, y1 in bs]
        labels[n] = [(name, b) for name, b in labels.get(n, []) if name != "outsole"] + new
        reviewed.add(n)
        n_box += len(new)
        n_changed += old != sorted(round(v) for _, b in new for v in b)
    print(f"{RELABEL.name}: 사진 {len(rl['images'])}장 → outsole 박스 {n_box}개 (바뀐 사진 {n_changed}장) | 기준: {rl.get('rule', '')}")

unknown = sorted({name for bs in labels.values() for name, _ in bs} - set(CLASSES) - {"not_sneakers"})
assert not unknown, f"모르는 라벨이 있습니다: {unknown}. CLASSES를 고치세요."

boxes = {}                                   # file_name -> [(cls, xc, yc, w, h)] (0~1 비율)
dropped = Counter()
for n in sorted(reviewed):
    W, H = size[n]
    if any(name == "not_sneakers" for name, _ in labels[n]):   # 운동화가 아닌 사진은 빼고 학습합니다.
        dropped["not_sneakers 사진"] += 1
        continue
    if n in unsure and not any(name == "outsole" for name, _ in labels[n]):
        dropped["판단 불가(재라벨 uncertain)"] += 1
        continue
    out = []
    for name, (x, y, w, h) in labels[n]:
        x0, y0, x1, y1 = max(x, 0), max(y, 0), min(x + w, W), min(y + h, H)
        if x1 - x0 < 4 or y1 - y0 < 4:
            dropped["너무 작은 박스"] += 1
            continue
        out.append((CLASSES.index(name), (x0 + x1) / 2 / W, (y0 + y1) / 2 / H, (x1 - x0) / W, (y1 - y0) / H))
    boxes[n] = out

n_pos = sum(1 for b in boxes.values() if b)
print(f"\n검수 구간 {len(runs)}개 + 확인된 job → 검수된 사진 {len(boxes)}장 (파손 박스 있음 {n_pos} / 없음 {len(boxes) - n_pos})")
print(f"제외: 검수 안 된 사진 {len(size) - len(reviewed)}장, {dict(dropped)}")
print("월별 검수된 사진:", dict(sorted(Counter(n[:6] for n in boxes).items())))
print("박스 수:", {c: sum(b[0] == i for bs in boxes.values() for b in bs) for i, c in enumerate(CLASSES)})
for job, (lo, hi) in KNOWN_REVIEWED.items():    # CVAT 화면의 박스 수와 같은지 확인하세요.
    names = [n for n in boxes if lo <= n <= hi]
    print(f"{job}: 사진 {len(names)}장, 박스 {sum(len(boxes[n]) for n in names)}개")

In [ ]:
# 5. 세션 단위 train / val 분리. YOLO 학습의 split.json이 있으면 그대로 써서 YOLO와 같은 사진으로 학습·채점합니다.
# 세션을 (밑창 박스 있음, 안감 박스 있음) 네 종류로 나누고 종류마다 세션의 VAL_RATIO만큼을 val로 보냅니다.
# 그래야 적은 밑창 클래스도 val에 고르게 들어갑니다. 분리 목록은 Drive에 저장하고, RESUME 때는 그대로 다시 씁니다.
import random
from datetime import datetime


def taken_at(name):
    return datetime.strptime(name[:15], "%Y%m%d_%H%M%S")


# 세션은 검수 안 된 사진까지 포함한 전체 촬영 순서로 끊습니다.
all_names = sorted(size, key=lambda n: (taken_at(n), n))
session_of, sid, prev = {}, 0, None
for n in all_names:
    t = taken_at(n)
    if prev is None or (t - prev).total_seconds() > SESSION_GAP:
        sid += 1
    session_of[n] = sid
    prev = t

SPLIT_JSON = OUT_DIR / "runs_rfdetr" / NAME / "split.json"
YOLO_SPLIT = OUT_DIR / "runs" / YOLO_RUN / "split.json"
inputs_now = [main_path.name] + [p.name for p, _, _ in overrides] + ([RELABEL.name] if RELABEL else [])
use_split = YOLO_SPLIT if (YOLO_SPLIT.exists() and not QUICK_TEST) else (SPLIT_JSON if RESUME and SPLIT_JSON.exists() else None)
if use_split:
    SPLIT_JSON = use_split
    saved = json.loads(SPLIT_JSON.read_text())
    train_names = [n for n in saved["train"] if n in boxes]
    val_names = [n for n in saved["val"] if n in boxes]
    print(f"저장된 분리를 다시 씁니다: {SPLIT_JSON}")
    if saved["inputs"] != inputs_now:
        print("⚠️ 처음 학습 때와 CVAT 파일이 다릅니다:", saved["inputs"], "→", inputs_now)
else:
    sessions = defaultdict(list)
    for n in boxes:
        sessions[session_of[n]].append(n)
    strata = defaultdict(list)
    for s, names in sessions.items():
        has = {b[0] for n in names for b in boxes[n]}
        strata[(0 in has, 1 in has)].append(s)

    rng = random.Random(SEED)
    prev_split = OUT_DIR / "runs" / SPLIT_FROM / "split.json"
    old_val, old_train = set(), set()
    if SPLIT_FROM and prev_split.exists() and not QUICK_TEST:
        prev = json.loads(prev_split.read_text())
        old_val = {session_of[n] for n in prev["val"] if n in session_of}
        old_train = {session_of[n] for n in prev["train"] if n in session_of} - old_val
    # 첫 학습의 검증 세션은 그대로 val, 첫 학습에 쓴 세션은 train, 나머지(새로 검수된 세션)만 층마다 VAL_RATIO씩 val로.
    val_sessions = set(old_val)
    for key in sorted(strata):
        ss = sorted(s for s in strata[key] if s not in old_val and s not in old_train)
        rng.shuffle(ss)
        val_sessions |= set(ss[:round(len(ss) * VAL_RATIO)])
    if old_val:
        print(f"{SPLIT_FROM}의 검증 세션 {len(old_val)}개 유지 + 새로 검수된 세션에서 {len(val_sessions) - len(old_val)}개 추가"
              f" (그 학습에 쓴 세션 {len(old_train)}개는 val에 넣지 않음)")

    val_names = sorted(n for n in boxes if session_of[n] in val_sessions)
    pos = sorted(n for n in boxes if session_of[n] not in val_sessions and boxes[n])
    neg_all = sorted(n for n in boxes if session_of[n] not in val_sessions and not boxes[n])
    n_neg = min(len(neg_all), round(len(pos) * NEG_PER_POS))
    # 음성은 재라벨에서 '파손 아님'으로 확인한 사진부터: 지운 오답(뒷굽 닳음·구멍 등) → 같은 신발의 다른 사진 → 나머지 무작위.
    tiers = [[n for n in neg_all if hard_neg.get(n) == "deleted"], [n for n in neg_all if hard_neg.get(n) == "none"],
             [n for n in neg_all if n not in hard_neg]]
    neg = []
    for tier in tiers:
        take = min(len(tier), n_neg - len(neg))
        neg += rng.sample(tier, take)
    neg = sorted(neg)
    print(f"음성 {len(neg)}장 = 지운 오답 {sum(hard_neg.get(n) == 'deleted' for n in neg)} + 같은 신발 확인 "
          f"{sum(hard_neg.get(n) == 'none' for n in neg)} + 무작위 {sum(n not in hard_neg for n in neg)}")
    if QUICK_TEST:
        pos, neg = pos[:90], neg[:30]
        val_names = [n for n in val_names if boxes[n]][:40] + [n for n in val_names if not boxes[n]][:20]
    train_names = sorted(pos + neg)
    print(f"세션 {len(sessions)}개 → val {len(val_sessions)}개 (train 파손 없음 후보 {len(neg_all)}장 중 {len(neg)}장 뽑음)")
    SPLIT_JSON.parent.mkdir(parents=True, exist_ok=True)
    SPLIT_JSON.write_text(json.dumps({"inputs": inputs_now, "train": train_names, "val": val_names}))

split = {**{n: "train" for n in train_names}, **{n: "val" for n in val_names}}
assert not {session_of[n] for n in train_names} & {session_of[n] for n in val_names}, "같은 세션이 train과 val에 섞였습니다."

print(f"\n{'':<14}{'train':>8}{'val':>8}")
print(f"{'사진':<14}{len(train_names):>8}{len(val_names):>8}")
print(f"{'  파손 없음':<14}{sum(not boxes[n] for n in train_names):>8}{sum(not boxes[n] for n in val_names):>8}")
for i, c in enumerate(CLASSES):
    ti = sum(any(b[0] == i for b in boxes[n]) for n in train_names)
    vi = sum(any(b[0] == i for b in boxes[n]) for n in val_names)
    tb = sum(b[0] == i for n in train_names for b in boxes[n])
    vb = sum(b[0] == i for n in val_names for b in boxes[n])
    print(f"{c:<14}{ti:>8}{vi:>8}   사진 / 박스 {tb} · {vb}")

In [ ]:
# 6. S3 원본을 받아 STORE_SIZE로 줄여 저장. 한 번 받은 사진은 Drive의 zip 한 개로 저장해 다음부터 다시 받지 않습니다.
import shutil
import time
from concurrent.futures import ThreadPoolExecutor

import boto3
from botocore.config import Config
from PIL import Image, ImageOps

IMG_DIR = ROOT / f"img{STORE_SIZE}"
IMG_DIR.mkdir(exist_ok=True)
CACHE_ZIP = OUT_DIR / f"img{STORE_SIZE}.zip"

need = sorted(split)
have = {p.name for p in IMG_DIR.glob("*.jpg")}
if CACHE_ZIP.exists() and not set(need) <= have:
    t0 = time.time()
    local = ROOT / CACHE_ZIP.name
    shutil.copy(CACHE_ZIP, local)            # Drive에서 바로 풀면 느려서 로컬로 복사한 뒤 풉니다.
    try:
        with zipfile.ZipFile(local) as z:
            z.extractall(IMG_DIR)
    except zipfile.BadZipFile:
        print("Drive 캐시 zip이 깨져 있어 S3에서 다시 받습니다.")
    local.unlink()
    have = {p.name for p in IMG_DIR.glob("*.jpg")}
    print(f"Drive 캐시에서 {len(have)}장 ({time.time() - t0:.0f}초)")

todo = [n for n in need if n not in have]
s3 = boto3.client("s3", region_name=REGION, aws_access_key_id=AWS_KEY[0], aws_secret_access_key=AWS_KEY[1],
                  config=Config(max_pool_connections=32, retries={"max_attempts": 8, "mode": "adaptive"}))


def fetch(name):
    body = s3.get_object(Bucket=BUCKET, Key=name)["Body"].read()
    im = Image.open(io.BytesIO(body))
    im.draft("RGB", (STORE_SIZE, STORE_SIZE))   # JPEG를 1/2 크기로 바로 풀어 빠르게 줄입니다.
    im = ImageOps.exif_transpose(im).convert("RGB")
    im.thumbnail((STORE_SIZE, STORE_SIZE), Image.LANCZOS)
    tmp = IMG_DIR / (name + ".part")
    im.save(tmp, "JPEG", quality=90)
    tmp.rename(IMG_DIR / name)
    return len(body)


if todo:
    t0, done, nbytes, failed = time.time(), 0, 0, []
    with ThreadPoolExecutor(24) as ex:
        futs = {ex.submit(fetch, n): n for n in todo}
        for f in futs:
            try:
                nbytes += f.result()
            except Exception as e:           # 받기 실패한 사진은 학습에서 뺍니다.
                failed.append((futs[f], repr(e)[:120]))
            done += 1
            if done % 500 == 0 or done == len(todo):
                el = time.time() - t0
                print(f"  {done}/{len(todo)}장 {nbytes / 1e9:.1f}GB {el:.0f}초 (남은 시간 약 {el / done * (len(todo) - done) / 60:.0f}분)")
    if failed:
        print(f"받기 실패 {len(failed)}장 — 학습에서 뺍니다. 예:", failed[:3])
        assert len(failed) < 0.05 * len(todo), "실패가 너무 많습니다. 키·인터넷을 확인하고 이 셀을 다시 실행하세요."
        for n, _ in failed:
            split.pop(n, None)
    if not QUICK_TEST and len(failed) < len(todo):
        # 캐시 zip 다시 쓰기 (사진은 이미 JPEG라 압축 없이 묶기만). 다 쓴 뒤에 이름을 바꿔 깨진 zip이 남지 않게 합니다.
        tmp_zip = ROOT / "cache.zip"
        with zipfile.ZipFile(tmp_zip, "w", zipfile.ZIP_STORED) as z:
            for p in sorted(IMG_DIR.glob("*.jpg")):
                z.write(p, p.name)
        part = CACHE_ZIP.with_name(CACHE_ZIP.name + ".part")
        shutil.copy(tmp_zip, part)
        os.replace(part, CACHE_ZIP)
        tmp_zip.unlink()
        print("Drive 캐시 저장:", CACHE_ZIP, f"{CACHE_ZIP.stat().st_size / 1e9:.2f}GB")

train_names = [n for n in train_names if n in split]
val_names = [n for n in val_names if n in split]
print(f"준비 완료: train {len(train_names)}장 / val {len(val_names)}장")

In [ ]:
# 7. RF-DETR용 COCO 폴더(train / valid). 사진은 심볼릭 링크로 걸고, 박스는 1280px 사진의 픽셀 좌표로 씁니다.
# 파손 없는 사진은 박스 없는 이미지로 넣어 배경으로 학습합니다.
from PIL import Image

DS = ROOT / "rfdetr_damage"
if DS.exists():
    shutil.rmtree(DS)
CATS = [{"id": i, "name": c, "supercategory": "damage"} for i, c in enumerate(CLASSES)]
img_size = {}
for part, names in (("train", train_names), ("valid", val_names)):
    d = DS / part
    d.mkdir(parents=True)
    imgs, anns = [], []
    # train에서는 outsole 사진을 이름만 바꾼 링크로 REPEAT_OUTSOLE번 넣습니다(YOLO 노트북과 같은 반복 샘플링).
    entries = [(n, n) for n in names]
    if part == "train":
        entries += [(n, f"{Path(n).stem}__r{r}.jpg") for n in names if any(b[0] == 0 for b in boxes[n])
                    for r in range(1, REPEAT_OUTSOLE)]
    for k, (n, fname) in enumerate(entries, 1):
        os.symlink(IMG_DIR / n, d / fname)
        if n not in img_size:
            with Image.open(IMG_DIR / n) as im:
                img_size[n] = im.size
        W, H = img_size[n]
        imgs.append({"id": k, "file_name": fname, "width": W, "height": H})
        for c, xc, yc, w, h in boxes[n]:
            anns.append({"id": len(anns) + 1, "image_id": k, "category_id": c, "iscrowd": 0,
                         "bbox": [(xc - w / 2) * W, (yc - h / 2) * H, w * W, h * H], "area": w * W * h * H})
    (d / "_annotations.coco.json").write_text(json.dumps({"images": imgs, "annotations": anns, "categories": CATS}))
    print(f"{part}: 사진 {len(imgs)}장, 박스 {len(anns)}개")

In [ ]:
# 8. 학습. 학습 중에는 검증하지 않고(eval_interval = EPOCHS → 마지막 epoch에만 채점) 마지막 EMA 가중치로 채점합니다.
# 배치는 T4 메모리에 맞게 자동으로 정하고(batch_size="auto", 실효 배치 16), T4는 fp16으로 학습합니다.
# 끊겼다면 3번 셀에서 RESUME = True로 바꾸고 모두 실행하면 마지막 epoch(last.ckpt)부터 이어 갑니다.
import time

from rfdetr import RFDETR, RFDETRMedium, RFDETRSmall

RUN = OUT_DIR / "runs_rfdetr" / NAME
FINAL = RUN / "last_ema.pth"                # 학습이 끝나면 생기는 마지막 EMA 가중치
Model = {"small": RFDETRSmall, "medium": RFDETRMedium}[RF_MODEL]
train_kwargs = dict(dataset_dir=str(DS), output_dir=str(RUN), epochs=EPOCHS, batch_size="auto", resolution=RESOLUTION,
                    eval_interval=EPOCHS, amp_dtype="fp16", num_workers=2, seed=SEED, tensorboard=False,
                    progress_bar="tqdm")

t0 = time.time()
if FINAL.exists():
    print("이미 끝난 학습입니다. 그대로 9번부터 채점합니다.")
elif RESUME and (RUN / "last.ckpt").exists():
    print("last.ckpt부터 이어 갑니다.")
    Model().train(**train_kwargs, resume=str(RUN / "last.ckpt"))
else:
    assert not RESUME, f"이어 갈 학습이 없습니다: {RUN / 'last.ckpt'}"
    RUN.mkdir(parents=True, exist_ok=True)
    Model().train(**train_kwargs)
print(f"학습 {(time.time() - t0) / 60:.0f}분 | 모델: {FINAL}")
assert FINAL.exists(), "last_ema.pth가 없습니다. 위 학습 로그를 확인하세요."

In [ ]:
# 9. 같은 검증 사진 · 같은 정답 · 같은 채점 코드로 RF-DETR과 YOLO26을 나란히 채점합니다.
#  - AP50 / AP50-95: 박스 단위 정확도(COCO 방식 101점 보간, 같은 클래스·한 정답에 한 예측)
#  - 파손 사진 찾음 / 잘못 경보: 사진 한 장 단위(앱이 "추가 발견"을 띄울지 정하는 기준)
#  - 크기별 찾은 비율: 정답 박스가 사진 한 변의 8% 미만(멀리서 찍은 작은 파손) / 8~20% / 20% 이상(근접 사진)
#  - 차이의 90% 구간: 검증 세션을 1,000번 다시 뽑아(세션 단위 부트스트랩) 계산. 0을 걸치면 우연일 수 있는 차이입니다.
import gc

import numpy as np
import pandas as pd
from IPython.display import display

PRED_BATCH = 8
THRS = np.arange(0.5, 0.96, 0.05)
LOOSE = 0.3                                  # 넉넉한 박스라 범위가 사람마다 다른 outsole은 IoU 0.3 기준도 같이 봅니다.
gc.collect()
torch.cuda.empty_cache()


def predict_rfdetr(names):
    net = RFDETR.from_checkpoint(str(FINAL), trust_checkpoint=True)
    assert set(CLASSES) <= set(net.class_names), f"RF-DETR 클래스 이름이 다릅니다: {net.class_names}"
    name_to_cls = {nm: CLASSES.index(nm) for nm in CLASSES}
    out = {}
    for i in range(0, len(names), PRED_BATCH):
        chunk = names[i:i + PRED_BATCH]
        dets = net.predict([str(IMG_DIR / n) for n in chunk], threshold=0.01, include_source_image=False)
        dets = dets if isinstance(dets, list) else [dets]
        for n, det in zip(chunk, dets):
            W, H = img_size[n]
            rows = []
            # predict()가 붙여 주는 이름을 씁니다. '물체 없음' 칸(__background__)은 여기서 걸러집니다.
            for xyxy, s, nm in zip(det.xyxy, det.confidence, det.data["class_name"]):
                if nm in name_to_cls:
                    rows.append((name_to_cls[nm], float(s), (xyxy[0] / W, xyxy[1] / H, xyxy[2] / W, xyxy[3] / H)))
            out[n] = rows
    del net
    return out


def predict_yolo(names, weights):
    from ultralytics import YOLO
    net = YOLO(str(weights))
    out = {}
    for i in range(0, len(names), PRED_BATCH):
        chunk = [str(IMG_DIR / n) for n in names[i:i + PRED_BATCH]]
        for res in net.predict(chunk, imgsz=YOLO_IMGSZ, conf=0.01, verbose=False):
            b = res.boxes
            out[Path(res.path).name] = [(int(c), float(s), tuple(map(float, xy)))
                                        for c, s, xy in zip(b.cls.tolist(), b.conf.tolist(), b.xyxyn.tolist())]
    del net
    return out


def gt_xyxy(n):
    return [(c, (xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2)) for c, xc, yc, w, h in boxes[n]]


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)


def match(preds, n, ci, thr):
    """그 사진의 클래스 ci 예측마다 (conf, 맞음?) — 높은 conf부터 아직 안 맞힌 정답과 IoU ≥ thr면 맞음."""
    g = [b for c, b in gt_xyxy(n) if c == ci]
    used = [False] * len(g)
    out = []
    for c, s, p in sorted([q for q in preds[n] if q[0] == ci], key=lambda q: -q[1]):
        best, bj = thr, -1
        for j, gb in enumerate(g):
            v = iou(p, gb)
            if not used[j] and v >= best:
                best, bj = v, j
        if bj >= 0:
            used[bj] = True
        out.append((s, bj >= 0))
    return out, len(g)


def ap_from(conf, tp, img, n_gt_img, w):
    """가중치 w(사진별 부트스트랩 횟수)로 COCO 101점 AP."""
    if len(conf) == 0:
        return 0.0
    order = np.argsort(-conf, kind="stable")
    ww = w[img[order]]
    ctp = np.cumsum(ww * tp[order])
    cfp = np.cumsum(ww * (1 - tp[order]))
    total = float((w * n_gt_img).sum())
    if total == 0:
        return float("nan")
    rec = ctp / total
    prec = ctp / np.maximum(ctp + cfp, 1e-9)
    prec = np.maximum.accumulate(prec[::-1])[::-1]
    idx = np.searchsorted(rec, np.linspace(0, 1, 101), side="left")
    return float(np.mean([prec[i] if i < len(prec) else 0.0 for i in idx]))


def tables(preds, names):
    """클래스별·IoU 문턱별 (conf, tp, 사진 번호) 배열과 사진별 정답 수."""
    T = {}
    for ci in range(len(CLASSES)):
        for thr in list(THRS) + [LOOSE]:
            conf, tp, img, ngt = [], [], [], np.zeros(len(names))
            for k, n in enumerate(names):
                m, g = match(preds, n, ci, thr)
                ngt[k] = g
                for s, ok in m:
                    conf.append(s); tp.append(1.0 if ok else 0.0); img.append(k)
            T[(ci, round(float(thr), 2))] = (np.array(conf), np.array(tp), np.array(img, dtype=int), ngt)
    return T


def image_level(preds, names, t):
    rows = {}
    clean = [n for n in names if not boxes[n]]
    for ci, cname in enumerate(CLASSES):
        has = [n for n in names if any(c == ci for c, _ in gt_xyxy(n))]
        hs = set(has)
        score = {n: max([s for c, s, _ in preds[n] if c == ci], default=0.0) for n in names}
        found = sum(score[n] >= t for n in has)
        fa = sum(score[n] >= t for n in clean)
        size_hit = {"작은(<8%)": [0, 0], "중간(8~20%)": [0, 0], "큰(≥20%)": [0, 0]}
        for n in has:
            pb = [p for c, s, p in preds[n] if c == ci and s >= t]
            for c, g in gt_xyxy(n):
                if c != ci:
                    continue
                side = ((g[2] - g[0]) * (g[3] - g[1])) ** 0.5
                key = "작은(<8%)" if side < 0.08 else "중간(8~20%)" if side < 0.2 else "큰(≥20%)"
                size_hit[key][1] += 1
                size_hit[key][0] += any(iou(g, p) >= 0.3 for p in pb)
        rows[cname] = {"파손 사진 찾음": f"{found}/{len(has)}", "잘못 경보(파손 없는 사진)": f"{fa}/{len(clean)}",
                       **{f"박스 찾음 {k}": f"{a}/{b}" for k, (a, b) in size_hit.items()}}
    return rows


names = list(val_names)
models = {"RF-DETR": predict_rfdetr(names)}
yolo_w = OUT_DIR / "runs" / YOLO_RUN / "weights" / "last.pt"
if yolo_w.exists() and not QUICK_TEST:
    models["YOLO26"] = predict_yolo(names, yolo_w)
else:
    print(f"YOLO 모델이 없어 RF-DETR만 채점합니다: {yolo_w}")

TB = {m: tables(p, names) for m, p in models.items()}
ones = np.ones(len(names))
rows = []
for m in models:
    r = {"모델": m}
    for ci, cname in enumerate(CLASSES):
        ap50 = ap_from(*TB[m][(ci, 0.5)][:3], TB[m][(ci, 0.5)][3], ones)
        ap = np.nanmean([ap_from(*TB[m][(ci, round(float(t), 2))][:3], TB[m][(ci, round(float(t), 2))][3], ones) for t in THRS])
        r[f"{cname} AP30"] = round(ap_from(*TB[m][(ci, LOOSE)][:3], TB[m][(ci, LOOSE)][3], ones), 3)
        r[f"{cname} AP50"] = round(ap50, 3)
        r[f"{cname} AP50-95"] = round(float(ap), 3)
    rows.append(r)
display(pd.DataFrame(rows))

def session_level(preds, names, t):
    """신발(세션) 단위: 같은 세션 사진 중 한 장이라도 그 클래스 박스(conf ≥ t)를 내면 찾음."""
    by_s = {}
    for n in names:
        by_s.setdefault(session_of[n], []).append(n)
    out = {}
    for ci, cname in enumerate(CLASSES):
        top = {n: max([s for c, s, _ in preds[n] if c == ci], default=0.0) for n in names}
        pos_s = [s for s, ns in by_s.items() if any(c == ci for n in ns for c, _ in gt_xyxy(n))]
        neg_s = [s for s, ns in by_s.items() if all(not boxes[n] for n in ns)]
        hit = sum(any(top[n] >= t for n in by_s[s]) for s in pos_s)
        fa = sum(any(top[n] >= t for n in by_s[s]) for s in neg_s)
        out[cname] = {"신발(세션) 찾음": f"{hit}/{len(pos_s)}", "잘못 경보(파손 없는 세션)": f"{fa}/{len(neg_s)}"}
    return out


for t in (0.25, 0.5):
    print(f"\n사진 단위 (conf ≥ {t})")
    display(pd.DataFrame({f"{m} · {c}": v for m, p in models.items() for c, v in image_level(p, names, t).items()}))
    print(f"신발(세션) 단위 (conf ≥ {t}) — 앱은 한 켤레를 여러 장 찍어 올리므로 이 숫자가 실제 사용에 가깝습니다")
    display(pd.DataFrame({f"{m} · {c}": v for m, p in models.items() for c, v in session_level(p, names, t).items()}))

# 세션 단위 부트스트랩: RF-DETR − YOLO26 AP50 차이의 90% 구간
summary = {"val_images": len(names), "rf_model": RF_MODEL, "resolution": RESOLUTION, "epochs": EPOCHS,
           "yolo_run": YOLO_RUN if "YOLO26" in models else None, "box": rows,
           "image_level": {m: {str(t): image_level(p, names, t) for t in (0.25, 0.5)} for m, p in models.items()},
           "session_level": {m: {str(t): session_level(p, names, t) for t in (0.25, 0.5)} for m, p in models.items()}}
if "YOLO26" in models:
    rng = np.random.default_rng(SEED)
    sess = np.array([session_of[n] for n in names])
    uniq = np.unique(sess)
    pos = {s: np.where(sess == s)[0] for s in uniq}
    diffs = {c: [] for c in CLASSES}
    for _ in range(1000):
        draw = rng.choice(uniq, size=len(uniq), replace=True)
        w = np.zeros(len(names))
        for s in draw:
            w[pos[s]] += 1
        for ci, c in enumerate(CLASSES):
            a = ap_from(*TB["RF-DETR"][(ci, 0.5)][:3], TB["RF-DETR"][(ci, 0.5)][3], w)
            b = ap_from(*TB["YOLO26"][(ci, 0.5)][:3], TB["YOLO26"][(ci, 0.5)][3], w)
            diffs[c].append(a - b)
    summary["ap50_diff_rfdetr_minus_yolo_90ci"] = {}
    for c in CLASSES:
        lo, hi = np.nanpercentile(diffs[c], [5, 95])
        summary["ap50_diff_rfdetr_minus_yolo_90ci"][c] = [round(float(lo), 3), round(float(hi), 3)]
        verdict = "RF-DETR이 낫다" if lo > 0 else "YOLO26이 낫다" if hi < 0 else "차이가 우연일 수 있음"
        print(f"{c}: AP50 차이(RF-DETR − YOLO26) 90% 구간 [{lo:+.3f}, {hi:+.3f}] → {verdict}")

(RUN / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1))
print("\n저장:", RUN / "summary.json")
print("아래 한 줄을 복사해 Claude에게 보내면 결과를 같이 볼 수 있습니다:\n")
print(json.dumps(summary, ensure_ascii=False, separators=(",", ":")))

In [ ]:
# 10. 두 모델이 엇갈린 사진 (초록 = 정답, 파랑 = RF-DETR, 빨강 = YOLO26, conf ≥ 0.25).
import cv2
import matplotlib.pyplot as plt

SHOW_CONF = 0.25
COL = {"RF-DETR": (40, 90, 255), "YOLO26": (255, 40, 40)}


def found(m, n):
    return any(any(iou(g, p) >= 0.3 for c2, s, p in models[m][n] if c2 == c and s >= SHOW_CONF) for c, g in gt_xyxy(n))


def draw(n):
    im = cv2.imread(str(IMG_DIR / n))[:, :, ::-1].copy()
    H, W = im.shape[:2]
    for c, g in gt_xyxy(n):
        cv2.rectangle(im, (int(g[0] * W), int(g[1] * H)), (int(g[2] * W), int(g[3] * H)), (0, 220, 0), 7)
    for m, p in models.items():
        for c, s, b in p[n]:
            if s >= SHOW_CONF:
                cv2.rectangle(im, (int(b[0] * W), int(b[1] * H)), (int(b[2] * W), int(b[3] * H)), COL[m], 4)
    return im


dmg = [n for n in names if boxes[n]]
if len(models) == 2:
    only_rf = [n for n in dmg if found("RF-DETR", n) and not found("YOLO26", n)][:6]
    only_yolo = [n for n in dmg if found("YOLO26", n) and not found("RF-DETR", n)][:6]
    print(f"RF-DETR만 찾은 파손 사진 {sum(found('RF-DETR', n) and not found('YOLO26', n) for n in dmg)}장 / "
          f"YOLO26만 찾은 사진 {sum(found('YOLO26', n) and not found('RF-DETR', n) for n in dmg)}장 (파손 사진 {len(dmg)}장 중)")
    groups = [("Only RF-DETR found", only_rf), ("Only YOLO26 found", only_yolo)]
else:
    groups = [("RF-DETR on damaged photos", dmg[:8])]
for title, ns_ in groups:
    if not ns_:
        continue
    cols = 3
    rows_ = (len(ns_) + cols - 1) // cols
    fig, axes = plt.subplots(rows_, cols, figsize=(5 * cols, 5 * rows_), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, n in zip(axes.flat, ns_):
        ax.imshow(draw(n))
        ax.set_title(n, fontsize=8)
    fig.suptitle(title)                      # Colab 기본 폰트에 한글이 없어 영어로 씁니다.
    plt.tight_layout()
    plt.show()

## 결과 읽는 법

- **AP50**: 박스가 정답과 절반 이상 겹치면 맞은 것으로 본 정확도(0~1). **AP50-95**는 더 꼭 맞아야 점수가 나옵니다.
- **사진 단위**: "파손 사진 찾음"은 높고 "잘못 경보"는 낮아야 앱의 AI 추가 발견에 쓸 수 있습니다.
- **크기별**: 고객이 신고한 파손은 근접 사진(10~15cm)을 받으므로 "큰(≥20%)"이 실제 주 사용처이고, "작은(<8%)"은 AI 추가 발견에 해당합니다.
- **90% 구간이 0을 걸치면** 두 모델의 차이는 이 검증 사진 수로는 가릴 수 없는 수준입니다. 그때는 라이선스(RF-DETR Apache-2.0 vs YOLO AGPL-3.0)와 속도로 고르면 됩니다.